# 17 — Patient-State Routing (Disease Classifier + Guideline-Locked Retrieval)

**The problem this fixes.** Many clinical questions arrive as a *patient story* rather than a
named disease — "a 58-year-old smoker with chest pain radiating to the left arm and sweating".
The AWMF corpus is organised by disease, so a raw patient-story query rarely lands on the right
guideline. An earlier attempt walked the UMLS/SNOMED finding graph to guess the disease, but on
these stories it proved unreliable: common findings (fatigue, disorientation, wheezing) fan out
to many diseases, so the graph mapped a diabetes story onto coronary disease, and specific
phrases ("chest pain radiating to the left arm") resolved to nothing.

**The approach here.** For the fixed set of ten GP diseases, the model itself is a far more
reliable router than the ontology. So the patient story is first **classified into one (or two)
of the ten diseases**, and retrieval is then **locked onto that disease's guideline**. The
patient state is transferred into the underlying condition before retrieval runs — the same goal
as before, done reliably.

**Pipeline.** question → classify into one of the ten diseases → HyDE translation (kept from the
earlier notebooks) enriched with that disease's German guideline terms → dense (pgvector) +
sparse (BM25), both restricted to the disease's guideline when the corpus carries a source tag →
Reciprocal Rank Fusion → cross-encoder rerank → grounded generation, with the story→disease
mapping handed to the model as a reasoning hint.

**Two design choices that matter for the scores.**
- The contexts scored by RAGAS are the reranked **AWMF chunks only**. The disease mapping is a
  generation hint, never injected into the retrieved context, so it cannot depress Context
  Precision.
- Retrieval is **guideline-locked** when the chunks carry a source/guideline tag in their
  metadata. A cell below inspects the metadata, auto-detects that tag, and matches each disease
  to its guideline. If no such tag exists, it falls back to strong German-term weighting.

**Prerequisites.** Same secrets as before (`NEON_DATABASE_URL`, `OPENROUTER_API_KEY`). The
UMLS key stays optional here — it is used only to enrich German disease terms, never for routing.

## Installs

In [ ]:
!pip install -q ragas langchain langchain-openai langchain-huggingface psycopg2-binary pgvector langchain-postgres datasets nest_asyncio sentence-transformers sqlalchemy requests rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 89.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.0/213.0 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 100.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 105.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB

## Vertex patch

A defensive shim so that importing parts of `langchain_community` does not pull in the Vertex
AI SDK, which is not installed in this environment.

In [ ]:
import sys, types
class DummyVertexAI: pass
class DummyChatVertexAI: pass
m = types.ModuleType("langchain_community.llms"); m.VertexAI = DummyVertexAI; sys.modules["langchain_community.llms"] = m
m = types.ModuleType("langchain_community.chat_models"); m.ChatVertexAI = DummyChatVertexAI; sys.modules["langchain_community.chat_models"] = m
m = types.ModuleType("langchain_community.chat_models.vertexai"); m.ChatVertexAI = DummyChatVertexAI; sys.modules["langchain_community.chat_models.vertexai"] = m
m = types.ModuleType("langchain_community.llms.vertexai"); m.VertexAI = DummyVertexAI; sys.modules["langchain_community.llms.vertexai"] = m

## Database, dense store, BM25 index, reranker, models

The same infrastructure as the earlier notebooks. Two additions: the BM25 loader now also reads
each chunk's `cmetadata` (so retrieval can be restricted to one guideline), and the answer
prompt receives a short "clinical mapping" line linking the patient story to the condition it
concerns.

In [ ]:
import os, json, time, random, re
import pandas as pd, torch, nest_asyncio
from google.colab import userdata, drive
from sqlalchemy import create_engine, text
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_postgres import PGVector
from langchain_core.prompts import PromptTemplate
from sentence_transformers import CrossEncoder
from rank_bm25 import BM25Okapi

nest_asyncio.apply()
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/'
df = pd.read_csv(DRIVE_PATH + 'AWMF_Golden_Dataset_200Q_Final.csv')

NEON = userdata.get('NEON_DATABASE_URL')
os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')

engine = create_engine(NEON, pool_pre_ping=True, pool_recycle=180, pool_size=2, max_overflow=2,
                       connect_args={"sslmode":"require","keepalives":1,"keepalives_idle":30})

COLLECTION = "awmf_baseline_bge"

# 1. Dense vector store (read-only; collection built in the earlier notebooks)
bge = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={'device':'cpu'})
vs = PGVector(embeddings=bge, collection_name=COLLECTION, connection=engine, use_jsonb=True)

# 2. Sparse BM25 index + chunk metadata (so retrieval can be locked to one guideline)
print("Loading texts + metadata for BM25 index...")
with engine.connect() as conn:
    res = conn.execute(text("""SELECT document, cmetadata FROM langchain_pg_embedding
                               WHERE collection_id = (SELECT uuid FROM langchain_pg_collection WHERE name = :c)"""),
                       {"c": COLLECTION})
    rows = res.fetchall()
all_texts = [r[0] for r in rows]
all_meta  = [(r[1] or {}) for r in rows]
tokenized_corpus = [doc.lower().split(" ") for doc in all_texts]
bm25_retriever = BM25Okapi(tokenized_corpus)
print(f"BM25 index built with {len(all_texts)} chunks.")

# 3. Cross-encoder reranker
print("Loading reranker...")
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device="cuda" if torch.cuda.is_available() else "cpu")

def make_llm(model, max_tokens=1024):
    return ChatOpenAI(model=model, api_key=os.environ["OPENROUTER_API_KEY"],
                      base_url="https://openrouter.ai/api/v1", temperature=0,
                      max_tokens=max_tokens, max_retries=6, request_timeout=90)

mistral = make_llm("mistralai/mistral-large")

def safe_invoke(llm, prompt, max_tries=8, base=8):
    for a in range(max_tries):
        try:
            return llm.invoke(prompt).content.strip()
        except Exception as e:
            if a == max_tries-1: raise
            w = min(base*(2**a)+random.uniform(0,3), 120)
            print(f"  retry {a+1}: {str(e)[:70]} ... {w:.0f}s"); time.sleep(w)

# HyDE translation, kept exactly as in the hybrid baseline.
query_gen_prompt = PromptTemplate(template="""You are a German medical expert.
1. First, precisely TRANSLATE the English question into German.
2. Second, write a 2-sentence hypothetical answer to the question in formal German clinical guideline terminology.
Do NOT use bullet points. Output ONLY the German translation followed directly by the German hypothetical answer.

English Question:
{question}""", input_variables=["question"])

# Disease classifier: transfer a patient story into one of the ten known conditions.
classify_prompt = PromptTemplate(template="""You are a clinical triage assistant.
A doctor describes a patient's presentation. Decide which of the following conditions the
description most likely concerns. Choose ONE; give a SECOND only if it is genuinely ambiguous.
Use ONLY names from this list, copied exactly:
{diseases}

Return strict JSON and nothing else: {{"diseases": ["<name>", "..."]}}

Patient presentation:
{question}""", input_variables=["diseases", "question"])

# Grounded answer prompt, now with the story->condition mapping as a reasoning hint.
qa_prompt = PromptTemplate(template="""You are an expert clinical assistant.
Answer the medical question in ENGLISH, grounded in the German AWMF guideline passages below.

The clinical mapping states which condition the patient's presentation most likely concerns;
use it to anchor the answer on the right condition, then answer from the guideline passages.
Keep the answer precise and concise. Do not invent citations.

Clinical mapping (patient presentation -> condition):
{mapping}

Context (German AWMF guidelines):
{context}

Question (English):
{question}

Answer (English):""", input_variables=["mapping", "context", "question"])
print("Setup complete.")

Mounted at /content/drive


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Setup complete.


## Inspect the chunk metadata and detect the guideline tag

To lock retrieval onto a disease's guideline, each chunk needs to remember which guideline it
came from. This cell prints a sample of `cmetadata` and, for every metadata key, how many
distinct values it takes. A key with a modest number of distinct short string values (roughly
one per guideline) is the source tag. The best candidate is auto-selected into `SOURCE_FIELD`;
if the guess is wrong, set `SOURCE_FIELD` by hand in the next cell.

In [ ]:
from collections import Counter

print("Sample cmetadata:")
for mm in all_meta[:3]:
    print(" ", mm)

keys = Counter()
for mm in all_meta:
    for k in mm.keys(): keys[k] += 1
print("\nMetadata keys (present in N chunks):", dict(keys))

def distinct_values(k):
    return [str(mm.get(k)) for mm in all_meta if mm.get(k) not in (None, "")]

print("\nPer-key distinct-value summary:")
candidates = []
for k in keys:
    uniq = sorted(set(distinct_values(k)))
    avg_len = (sum(len(v) for v in uniq) / len(uniq)) if uniq else 0
    print(f"  {k}: {len(uniq)} distinct, avg len {avg_len:.0f}")
    # a plausible guideline tag: several-to-dozens of short-ish labels, present on most chunks
    if 2 <= len(uniq) <= 60 and avg_len <= 120 and keys[k] >= 0.5 * len(all_meta):
        candidates.append((k, len(uniq)))

SOURCE_FIELD = candidates[0][0] if candidates else None
print("\nAuto-detected SOURCE_FIELD:", SOURCE_FIELD)
if SOURCE_FIELD:
    print("Its distinct values:")
    for v in sorted(set(distinct_values(SOURCE_FIELD)))[:60]:
        print("   -", v[:100])
else:
    print("No obvious guideline tag found -> retrieval will fall back to German-term weighting.")

Sample cmetadata:
  {'page': 218, 'title': 'Nationale VersorgungsLeitlinie Unipolare Depression – Langfassung, Version 3.2', 'author': 'Bundesärztekammer (BÄK);Kassenärztliche Bundesvereinigung (KBV);Arbeitsgemeinschaft der Wissenschaftlichen Medizinischen Fachgesellschaften (AWMF)', 'source': 'C:/Users/Aya Aidi/Documents/PMS project/PMS-data-testing-/backend/guidelines/chronic_most_common_diseases_for_GP\\depression-vers3-2-lang.pdf', 'company': 'ÄZQ', 'creator': 'Acrobat PDFMaker 17 für Word', 'moddate': '2023-07-05T11:37:28+02:00', 'subject': '', 'comments': '', 'keywords': '', 'producer': 'Adobe PDF Library 17.11.238', 'page_label': '219', 'total_pages': 256, 'creationdate': '2023-07-04T13:26:03+02:00', 'sourcemodified': 'D:20230704112445', 'citavidocumentproperty_6': 'False'}
  {'page': 218, 'title': 'Nationale VersorgungsLeitlinie Unipolare Depression – Langfassung, Version 3.2', 'author': 'Bundesärztekammer (BÄK);Kassenärztliche Bundesvereinigung (KBV);Arbeitsgemeinschaft der Wi

## The ten diseases, their German guideline terms, and guideline matching

The ten GP diseases (from the data-understanding analysis) with the German terms their AWMF/NVL
guidelines use. The German terms drive retrieval and, where a source tag exists, are matched
against the guideline labels so each disease knows exactly which chunks are its own. The UMLS key
is used here only to add a few more German synonyms, and is skipped silently if unavailable.

If `SOURCE_FIELD` was mis-detected above, set it correctly here (`SOURCE_FIELD = "your_key"`)
before the matching runs.

In [ ]:
# SOURCE_FIELD = "source"   # <- uncomment and set by hand only if the auto-detection was wrong

DISEASE_DB = [
    {"name": "Hypertension",          "de": ["Hypertonie", "arterielle Hypertonie", "Bluthochdruck"]},
    {"name": "Diabetes",              "de": ["Typ-2-Diabetes", "Diabetes mellitus", "Diabetes"]},
    {"name": "Asthma",                "de": ["Asthma bronchiale", "Asthma"]},
    {"name": "Depression",            "de": ["Depression", "unipolare Depression", "depressive Episode"]},
    {"name": "Chronic Back Pain",     "de": ["Kreuzschmerz", "chronischer Rückenschmerz", "Rückenschmerz"]},
    {"name": "Coronary Heart Disease","de": ["Koronare Herzkrankheit", "KHK", "chronisches Koronarsyndrom"]},
    {"name": "Osteoarthritis",        "de": ["Arthrose", "Gonarthrose", "Koxarthrose"]},
    {"name": "COPD",                  "de": ["COPD", "chronisch obstruktive Lungenerkrankung"]},
    {"name": "Dementia",              "de": ["Demenz", "Alzheimer"]},
    {"name": "Heart Failure",         "de": ["Herzinsuffizienz", "chronische Herzinsuffizienz"]},
]
DISEASE_NAMES = [d["name"] for d in DISEASE_DB]

# Optional: enrich German terms via UMLS (exact disease names only -> low noise). Harmless if the key is absent.
try:
    import requests, functools
    UMLS_API_KEY = userdata.get('UMLS_API_KEY')
    UTS = "https://uts-ws.nlm.nih.gov/rest"
    @functools.lru_cache(maxsize=256)
    def _cui(term):
        r = requests.get(f"{UTS}/search/current", params={"string": term, "apiKey": UMLS_API_KEY, "pageSize": 1}, timeout=15)
        hits = r.json().get("result", {}).get("results", []) if r.ok else []
        return hits[0]["ui"] if hits and hits[0].get("ui") not in (None, "NONE") else None
    @functools.lru_cache(maxsize=256)
    def _ger(cui):
        r = requests.get(f"{UTS}/content/current/CUI/{cui}/atoms", params={"language":"GER","apiKey":UMLS_API_KEY,"pageSize":20}, timeout=15)
        return [a.get("name","").strip() for a in (r.json().get("result", []) if r.ok else []) if a.get("name")]
    if UMLS_API_KEY:
        for d in DISEASE_DB:
            cui = _cui(d["name"])
            if cui:
                for g in _ger(cui)[:3]:
                    if g and all(g.lower() != x.lower() for x in d["de"]): d["de"].append(g)
        print("German terms enriched via UMLS.")
    else:
        print("No UMLS key -> using the hard-coded German terms only.")
except Exception as e:
    print("UMLS enrichment skipped:", str(e)[:80])

# Match each disease to the guideline sources that mention its name (English or German).
def build_disease_sources():
    if not SOURCE_FIELD: return {}
    vals = sorted(set(str(mm.get(SOURCE_FIELD)) for mm in all_meta if mm.get(SOURCE_FIELD)))
    d2s = {}
    for d in DISEASE_DB:
        terms = [d["name"]] + d["de"]
        d2s[d["name"]] = [v for v in vals if any(t.lower() in v.lower() for t in terms)]
    return d2s

DISEASE_SOURCES = build_disease_sources()
print("\nGuideline sources matched per disease:")
for d in DISEASE_NAMES:
    ms = DISEASE_SOURCES.get(d, [])
    print(f"  {d}: {len(ms)} source(s)" + (f" -> {ms[0][:60]}" if ms else ""))
LOCKED = SOURCE_FIELD is not None and any(DISEASE_SOURCES.values())
print("\nGuideline-locked retrieval:", "ON" if LOCKED else "OFF (falling back to German-term weighting)")

German terms enriched via UMLS.

Guideline sources matched per disease:
  Hypertension: 1 source(s) -> Nationale VersorgungsLeitlinie Hypertonie – Langfassung, Ver
  Diabetes: 1 source(s) -> Nationale VersorgungsLeitlinie Typ-2-Diabetes – Langfassung,
  Asthma: 1 source(s) -> Nationale VersorgungsLeitlinie Asthma – Langfassung, Version
  Depression: 1 source(s) -> Nationale VersorgungsLeitlinie Unipolare Depression – Langfa
  Chronic Back Pain: 1 source(s) -> Nationale VersorgungsLeitlinie Nicht-spezifischer Kreuzschme
  Coronary Heart Disease: 1 source(s) -> Nationale VersorgungsLeitlinie Chronische KHK, Version 7
  Osteoarthritis: 0 source(s)
  COPD: 1 source(s) -> Nationale VersorgungsLeitlinie COPD – Teilpublikation der La
  Dementia: 0 source(s)
  Heart Failure: 1 source(s) -> Nationale VersorgungsLeitlinie Chronische Herzinsuffizienz –

Guideline-locked retrieval: ON


## Classify the presentation, then retrieve from the matched guideline

`classify_disease` turns the patient story into one or two of the ten conditions. `retrieve`
then runs the usual HyDE + hybrid pipeline, but the dense and sparse candidate pools are
restricted to the classified disease's guideline whenever a source tag is available. HyDE is
unchanged; the contexts returned for scoring are the reranked AWMF chunks alone, while the
story→disease mapping is returned separately as a generation hint.

In [ ]:
src_of = {}
if SOURCE_FIELD:
    for t, mm in zip(all_texts, all_meta):
        src_of.setdefault(t, mm.get(SOURCE_FIELD))

def question_gist(q):
    q = q.strip().replace("\n", " ")
    return (q[:80] + "…") if len(q) > 80 else q

def classify_disease(question):
    raw = safe_invoke(mistral, classify_prompt.format(diseases="\n".join(DISEASE_NAMES), question=question))
    picks = []
    m = re.search(r"\{.*\}", raw, re.S)
    if m:
        try: picks = json.loads(m.group(0)).get("diseases", [])
        except Exception: picks = []
    if not picks:
        picks = [n for n in DISEASE_NAMES if n.lower() in raw.lower()]
    norm = []
    for p in picks:
        for n in DISEASE_NAMES:
            if str(p).strip().lower() == n.lower() and n not in norm: norm.append(n)
    return norm[:2]

def retrieve(question, k_final=10):
    # 1. HyDE translation (kept as before)
    hyde = safe_invoke(mistral, query_gen_prompt.format(question=question))

    # 2. Classify the patient story into the underlying condition(s)
    diseases = classify_disease(question)
    de_terms = [t for d in diseases for t in next((x["de"] for x in DISEASE_DB if x["name"] == d), [])]
    sources = [s for d in diseases for s in DISEASE_SOURCES.get(d, [])]
    mapping = " ; ".join(f"{question_gist(question)} -> {d}" for d in diseases) if diseases else "None."

    # 3. Enriched query = HyDE + the condition's German guideline terms
    enriched = f"{hyde} {' '.join(de_terms)}".strip()

    # 4. Dense retrieval, locked to the guideline when possible
    if sources and SOURCE_FIELD:
        dretr = vs.as_retriever(search_kwargs={"k": 30, "filter": {SOURCE_FIELD: {"$in": sources}}})
    else:
        dretr = vs.as_retriever(search_kwargs={"k": 30})
    dense_docs = [d.page_content for d in dretr.invoke(enriched)]

    # 5. Sparse retrieval, restricted to the same guideline when possible
    cand = bm25_retriever.get_top_n(enriched.lower().split(" "), all_texts, n=100)
    if sources and SOURCE_FIELD:
        allowed = set(sources)
        sparse_docs = [t for t in cand if src_of.get(t) in allowed][:30]
    else:
        sparse_docs = cand[:30]

    # 6. Reciprocal Rank Fusion
    rrf = {}
    for rank, doc in enumerate(dense_docs): rrf[doc] = rrf.get(doc, 0.0) + 1.0/(60+rank)
    for rank, doc in enumerate(sparse_docs): rrf[doc] = rrf.get(doc, 0.0) + 1.0/(60+rank)
    fused = [doc for doc, _ in sorted(rrf.items(), key=lambda x: x[1], reverse=True)[:30]]
    if not fused:  # a guideline filter that emptied the pool -> unfiltered dense fallback
        fused = [d.page_content for d in vs.as_retriever(search_kwargs={"k": 30}).invoke(enriched)]

    # 7. Cross-encoder rerank
    scores = reranker.predict([[enriched, t] for t in fused])
    reranked = [t for _, t in sorted(zip(scores, fused), key=lambda x: x[0], reverse=True)][:k_final]

    # Contexts scored by RAGAS = AWMF chunks only; the mapping is a generation hint.
    return reranked, mapping, diseases

# Smoke test
_q = df.iloc[0]['English_Open_Question']
_ctx, _map, _dis = retrieve(_q)
print("Question:", _q[:90])
print("Classified:", _dis)
print("Top passage source:", src_of.get(_ctx[0]) if SOURCE_FIELD else "(no source tag)")
print("Top passage:", _ctx[0][:180], "...")

Question: A 58-year-old man comes to the emergency department because of increasing shortness of bre
Classified: ['COPD', 'Heart Failure']
Top passage source: Nationale VersorgungsLeitlinie COPD – Teilpublikation der Langfassung, 2. Auflage. Version 1
Top passage: Halsvenenstauung, Lebervergrößerung, periphere Ödeme, Aszites 
Zeichen der Linksherzinsuffizienz  
(Rückstau des Blutes in die 
Lungengefäße): 
 Symptome: Atemnot, zunächst unter  ...


## Sanity check on patient-state questions

The four hand-written, state-phrased questions from before. Each should now be classified into
the right condition, and the top retrieved passage should come from that condition's guideline.

In [ ]:
demo_questions = [
    "A 58-year-old current smoker reports chest pain radiating to the left arm and sweating. What is the management?",
    "An obese patient with excessive thirst, frequent urination and fatigue. How should this be managed?",
    "An elderly patient with progressive memory loss, disorientation and difficulty finding words. What is the approach?",
    "A patient with wheezing, shortness of breath and a chronic morning cough who is a long-term smoker.",
]
for q in demo_questions:
    ctx, mapping, diseases = retrieve(q)
    print("Q:", q)
    print("  classified:", diseases)
    print("  top passage source:", src_of.get(ctx[0]) if SOURCE_FIELD else "(no source tag)")
    print("  top passage:", ctx[0][:150].replace(chr(10), " "), "...")
    print()

Q: A 58-year-old current smoker reports chest pain radiating to the left arm and sweating. What is the management?
  classified: ['Coronary Heart Disease']
  top passage source: Nationale VersorgungsLeitlinie Chronische KHK, Version 7
  top passage: NVL Chronische KHK  Langfassung – Version 7.0   © NVL-Programm 2024      Seite 19  Ursache des Brustschmerzes Häufigkeit Prozent  Erkrankungen der At ...

Q: An obese patient with excessive thirst, frequent urination and fatigue. How should this be managed?
  classified: ['Diabetes']
  top passage source: Nationale VersorgungsLeitlinie Typ-2-Diabetes – Langfassung, Version 3.0
  top passage: NVL Typ-2-Diabetes  Langfassung – Version 3.0   © NVL-Programm 2023  Seite 46  4.1.1 Anamnese und körperliche Untersuchungen  Empfehlung  4-2 | k | m ...

Q: An elderly patient with progressive memory loss, disorientation and difficulty finding words. What is the approach?
  classified: ['Dementia']
  top passage source: Nationale VersorgungsLeitlinie

## Generation loop

Every question is answered with the classify-then-retrieve pipeline. The story→disease mapping
is passed to the generator as a reasoning hint; the contexts stored for evaluation are the
reranked AWMF chunks alone. Results are written to Drive after each item so the run resumes
cleanly if the session drops.

In [ ]:
gt_df = pd.read_csv(DRIVE_PATH + "AWMF_CorpusGrounded_GroundTruth.csv")
gt_map = gt_df.set_index('English_Open_Question')['corpus_ground_truth'].to_dict()

work = df.reset_index(drop=True)  # full 200 questions
print(f"Generating routed answers for {len(work)} questions")

out_file = DRIVE_PATH + "PATIENT_STATE_ROUTED_results.json"
if os.path.exists(out_file):
    res = json.load(open(out_file)); done = set(res["question"])
    print(f"  resuming -- {len(done)} already done")
else:
    res = {"question":[],"answer":[],"contexts":[],"disease":[],
           "medqa_ground_truth":[],"corpus_ground_truth":[]}; done = set()

for i in range(len(work)):
    r = work.iloc[i]; q = r['English_Open_Question']
    if q in done: continue
    try:
        ctx, mapping, diseases = retrieve(q)
        ans = safe_invoke(mistral, qa_prompt.format(mapping=mapping, context="\n\n".join(ctx), question=q))

        res["question"].append(q); res["answer"].append(ans); res["contexts"].append(ctx)
        res["disease"].append(diseases)
        res["medqa_ground_truth"].append(r['English_Correct_Text'])
        res["corpus_ground_truth"].append(gt_map.get(q, "NOT_IN_CORPUS"))
        done.add(q)
        json.dump(res, open(out_file, "w"))

        if len(res["question"]) % 10 == 0: print(f"  {len(res['question'])}/{len(work)}")
        time.sleep(1.5)
    except Exception as e:
        print("err", i, str(e)[:100]); time.sleep(8)

print(f"Generation complete -> {out_file}")

Generating routed answers for 200 questions
  resuming -- 5 already done
  10/200
  20/200
  30/200
  retry 1: Error code: 429 - {'error': {'message': 'Provider returned error', 'co ... 10s
  40/200
  50/200
  60/200
  70/200
  80/200
  90/200
  100/200
  110/200
  120/200
  130/200
  140/200
  150/200
  160/200
  170/200
  180/200
  190/200
  200/200
Generation complete -> /content/drive/MyDrive/PATIENT_STATE_ROUTED_results.json


## Evaluation against both ground truths

The same GPT-4o-mini judge and the same four RAGAS metrics as every earlier notebook, so the
numbers are directly comparable. The corpus-grounded score covers the answerable subset; the
MedQA score covers the full benchmark.

In [ ]:
from datasets import Dataset, Features, Value, Sequence
from ragas import evaluate
from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig

judge = LangchainLLMWrapper(make_llm("openai/gpt-4o-mini", max_tokens=4096))
remb = LangchainEmbeddingsWrapper(bge)
feat = Features({"question":Value("string"),"answer":Value("string"),
                 "contexts":Sequence(Value("string")),"ground_truth":Value("string")})

def score(path, gt_key, keep=None):
    d = json.load(open(path))
    idx = [i for i in range(len(d["question"])) if keep is None or keep(d, i)]
    dd = {"question":[d["question"][i] for i in idx],
          "answer":[d["answer"][i] for i in idx],
          "contexts":[d["contexts"][i] for i in idx],
          "ground_truth":[d[gt_key][i] for i in idx]}
    ds = Dataset.from_dict(dd, features=feat)
    out = evaluate(ds, metrics=[context_precision, context_recall, faithfulness, answer_relevancy],
                   llm=judge, embeddings=remb, run_config=RunConfig(timeout=300, max_workers=2, max_retries=5))
    return len(idx), dict(out.to_pandas()[["context_precision","context_recall","faithfulness","answer_relevancy"]].mean().round(3))

in_corpus = lambda d, i: d["corpus_ground_truth"][i] not in (None, "", "NOT_IN_CORPUS")

print("=== PATIENT-STATE ROUTED SCORES ===")
n, s = score(out_file, "corpus_ground_truth", keep=in_corpus); print(f"Corpus-Grounded (answerable, n={n}):", s)
n, s = score(out_file, "medqa_ground_truth");                  print(f"Full 200 (MedQA GT, n={n}):        ", s)

/tmp/ipykernel_2980/251768042.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
/tmp/ipykernel_2980/251768042.py:3: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
/tmp/ipykernel_2980/251768042.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import context_precision

=== PATIENT-STATE ROUTED SCORES ===


Evaluating:   0%|          | 0/196 [00:00<?, ?it/s]

Corpus-Grounded (answerable, n=49): {'context_precision': np.float64(0.661), 'context_recall': np.float64(0.808), 'faithfulness': np.float64(0.475), 'answer_relevancy': np.float64(0.572)}


Evaluating:   0%|          | 0/800 [00:00<?, ?it/s]

Full 200 (MedQA GT, n=200):         {'context_precision': np.float64(0.144), 'context_recall': np.float64(0.22), 'faithfulness': np.float64(0.355), 'answer_relevancy': np.float64(0.529)}
